# 🌸 Classificação do Dataset Iris — Machine Learning
**FEMA/IMESA — 4º Bacharelado em Ciência da Computação**  
**Disciplina:** Inteligência Artificial | **Prof.:** Talo

---
Este notebook implementa dois modelos de classificação sobre o dataset Iris:
- **Modelo 1 (fixo):** Naive Bayes
- **Modelo 2 (sua escolha):** selecione no menu abaixo

Os modelos são treinados em **R** (via `rpy2`) e os resultados são exibidos com visualizações em Python.

## 1. Instalação de dependências

In [ ]:
# Instala rpy2 (ponte Python <-> R) e pacotes R necessários
import subprocess, sys

print('Instalando rpy2...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'rpy2', '--quiet'], check=True)

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

print('Instalando pacotes R (pode levar alguns minutos)...')
ro.r("""
pkgs <- c('e1071', 'caret', 'rpart', 'randomForest', 'kernlab', 'ggplot2')
new <- pkgs[!(pkgs %in% installed.packages()[,'Package'])]
if (length(new)) install.packages(new, repos='https://cloud.r-project.org', quiet=TRUE)
""")
print('✅ Tudo pronto!')

## 2. Menu — Escolha o 2º algoritmo

In [ ]:
import ipywidgets as widgets
from IPython.display import display

algoritmo_widget = widgets.RadioButtons(
    options=[
        ('🌳  Árvore de Decisão (rpart)', 'rpart'),
        ('🌲  Random Forest (randomForest)', 'rf'),
        ('⚡  SVM — Support Vector Machine (kernlab)', 'svm')
    ],
    description='2º Modelo:',
    layout=widgets.Layout(width='400px'),
    style={'description_width': '80px'}
)

botao = widgets.Button(
    description='▶  Executar análise',
    button_style='success',
    layout=widgets.Layout(width='200px', margin='10px 0 0 0')
)

output = widgets.Output()

display(algoritmo_widget, botao, output)

## 3. Engine de análise (Python + R)

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pandas2ri.activate()

NOMES_ALGORITMO = {
    'rpart': 'Árvore de Decisão',
    'rf':    'Random Forest',
    'svm':   'SVM'
}

# ── Código R parametrizado ─────────────────────────────────────────────────────
R_SCRIPT = """
# =============================================================================
# PACOTES UTILIZADOS
# =============================================================================
# Cada library() carrega um pacote instalado anteriormente.
# Pacotes são conjuntos de funções prontas — equivalente a 'import' no Python.

library(e1071)        # Contém naiveBayes() — implementação do Naive Bayes
library(caret)        # Contém createDataPartition() e confusionMatrix()
library(rpart)        # Contém rpart() — Árvore de Decisão
library(randomForest) # Contém randomForest() — ensemble de árvores
library(kernlab)      # Contém ksvm() — Support Vector Machine

# =============================================================================
# ETAPA 1 — COLETA E PREPARAÇÃO DOS DADOS
# =============================================================================

# data(iris) carrega o dataset Iris que já vem embutido no R.
# São 150 amostras com 4 atributos numéricos e 1 coluna de classe (Species).
data(iris)

# set.seed() fixa a semente do gerador de números aleatórios.
# Garante que a divisão treino/teste seja a mesma toda vez que o código rodar,
# tornando os resultados reproduzíveis.
set.seed(42)

# createDataPartition() divide os índices das linhas de forma ESTRATIFICADA:
#   p = 0.8  → 80% dos dados vão para treino, 20% para teste
#   list = FALSE → retorna um vetor simples de índices (não uma lista)
# Estratificada = a proporção de cada espécie é mantida nos dois conjuntos.
idx   <- createDataPartition(iris$Species, p = 0.8, list = FALSE)

# Usa os índices para separar o dataframe iris em dois subconjuntos:
#   iris[ idx, ]  → linhas cujos índices estão em idx  (treino)
#   iris[-idx, ]  → todas as outras linhas              (teste)
train <- iris[ idx, ]
test  <- iris[-idx, ]

cat(sprintf("Dataset: %d amostras | Treino: %d | Teste: %d\\n",
            nrow(iris), nrow(train), nrow(test)))

# =============================================================================
# ETAPA 2 — TREINAMENTO: NAIVE BAYES
# =============================================================================

# naiveBayes() treina o modelo usando o Teorema de Bayes com a suposição de
# independência condicional entre os atributos (por isso "Naive" = ingênuo).
#   Species ~ .  → fórmula: prever 'Species' usando TODOS os outros atributos (.)
#   data = train → usa apenas os dados de treino para aprender
nb_model  <- naiveBayes(Species ~ ., data = train)

# predict() aplica o modelo treinado sobre os dados de TESTE.
# O modelo nunca viu esses dados antes — é aqui que medimos a performance real.
nb_pred   <- predict(nb_model, newdata = test)

# confusionMatrix() compara as previsões com os valores reais (test$Species).
# Gera a matriz de confusão, acurácia, precisão, recall e outras métricas.
nb_cm     <- confusionMatrix(nb_pred, test$Species)

# Extrai apenas a Acurácia geral do objeto retornado por confusionMatrix().
# Acurácia = (acertos totais) / (total de amostras no teste)
nb_acc    <- nb_cm$overall['Accuracy']

cat("\\n=== NAIVE BAYES ===\\n")
print(nb_cm$table)  # Imprime a matriz de confusão (linhas = real, colunas = previsto)
cat(sprintf("Acurácia: %.4f (%.1f%%)\\n", nb_acc, nb_acc * 100))

# =============================================================================
# ETAPA 3 — TREINAMENTO: SEGUNDO ALGORITMO (escolhido no menu Python)
# =============================================================================
# A variável ALGO_CHOICE é injetada pelo Python antes de executar este script.
# O if/else seleciona qual algoritmo treinar com base na escolha do usuário.

if (ALGO_CHOICE == 'rpart') {

  # ----- ÁRVORE DE DECISÃO (rpart) -----
  # rpart() constrói uma árvore de decisão por particionamento recursivo.
  #   method = 'class' → problema de classificação (não regressão)
  # A árvore aprende regras do tipo:
  #   "se PetalLength > 2.45 → não é setosa; senão → é setosa"
  m2      <- rpart(Species ~ ., data = train, method = 'class')

  # type = 'class' faz o predict() retornar o nome da classe prevista.
  # Sem isso, retornaria uma matriz de probabilidades por classe.
  m2_pred <- predict(m2, newdata = test, type = 'class')

} else if (ALGO_CHOICE == 'rf') {

  # ----- RANDOM FOREST -----
  # randomForest() cria um ensemble (conjunto) de múltiplas árvores de decisão.
  #   ntree = 100 → treina 100 árvores independentes, cada uma com
  #                 uma amostra e subconjunto de atributos diferentes (bagging)
  # Cada árvore "vota" em uma classe; a classe mais votada é a previsão final.
  # Esse mecanismo reduz overfitting em relação a uma única árvore.
  m2      <- randomForest(Species ~ ., data = train, ntree = 100)

  # predict() para Random Forest retorna a classe diretamente
  m2_pred <- predict(m2, newdata = test)

} else if (ALGO_CHOICE == 'svm') {

  # ----- SVM — Support Vector Machine (kernlab) -----
  # ksvm() implementa o SVM com kernel para dados não linearmente separáveis.
  #   kernel = 'rbfdot' → kernel RBF (Radial Basis Function / Gaussiano):
  #     projeta os dados em um espaço de dimensão maior onde se tornam
  #     linearmente separáveis, sem calcular a transformação explicitamente
  #     (truque do kernel). É o kernel mais indicado para atributos contínuos.
  m2      <- ksvm(Species ~ ., data = train, kernel = 'rbfdot')

  m2_pred <- predict(m2, newdata = test)
}

# Avalia o segundo modelo da mesma forma que o Naive Bayes
m2_cm  <- confusionMatrix(m2_pred, test$Species)
m2_acc <- m2_cm$overall['Accuracy']

cat(sprintf("\\n=== %s ===\\n", toupper(ALGO_CHOICE)))
print(m2_cm$table)
cat(sprintf("Acurácia: %.4f (%.1f%%)\\n", m2_acc, m2_acc * 100))

# =============================================================================
# ETAPA 4 — EXPORTA RESULTADOS PARA O PYTHON
# =============================================================================
# as.vector() "achata" a matriz 3x3 em um vetor de 9 elementos para
# facilitar a transferência de volta ao ambiente Python via rpy2.
nb_matrix <- as.vector(nb_cm$table)  # 9 valores da matriz de confusão do NB
m2_matrix <- as.vector(m2_cm$table)  # 9 valores da matriz de confusão do M2
accs      <- c(nb_acc, m2_acc)       # vetor com as duas acurácias
"""

# ── Função que plota matrizes de confusão ──────────────────────────────────────
def plot_confusion(ax, matrix_flat, title, accuracy):
    labels = ['setosa', 'versicolor', 'virginica']
    mat = np.array(matrix_flat).reshape(3, 3)
    im = ax.imshow(mat, interpolation='nearest', cmap='Blues')
    ax.set_title(f'{title}\nAcurácia: {accuracy*100:.1f}%', fontsize=13, fontweight='bold')
    tick_marks = np.arange(3)
    ax.set_xticks(tick_marks); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticks(tick_marks); ax.set_yticklabels(labels)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_xlabel('Previsto', fontsize=11)
    thresh = mat.max() / 2
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(int(mat[i, j])),
                    ha='center', va='center',
                    color='white' if mat[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax)

# ── Callback do botão ─────────────────────────────────────────────────────────
def executar(b):
    with output:
        output.clear_output()
        escolha = algoritmo_widget.value
        nome    = NOMES_ALGORITMO[escolha]
        print(f'\n⏳ Executando: Naive Bayes  vs  {nome}...\n')

        # Injeta a escolha do usuário no ambiente R antes de rodar o script
        ro.r(f'ALGO_CHOICE <- "{escolha}"')
        ro.r(R_SCRIPT)

        # Recupera os resultados exportados pelo R (Etapa 4)
        nb_mat = list(ro.r('nb_matrix'))
        m2_mat = list(ro.r('m2_matrix'))
        accs   = list(ro.r('accs'))

        # Plota as duas matrizes de confusão lado a lado
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle('Matrizes de Confusão — Dataset Iris', fontsize=15, fontweight='bold', y=1.02)

        plot_confusion(axes[0], nb_mat, 'Naive Bayes', accs[0])
        plot_confusion(axes[1], m2_mat, nome,          accs[1])

        plt.tight_layout()
        plt.show()

        # ── Análise comparativa ───────────────────────────────────────────────
        print('\n' + '='*55)
        print('          ANÁLISE COMPARATIVA')
        print('='*55)
        print(f'  Naive Bayes   → Acurácia: {accs[0]*100:.1f}%')
        print(f'  {nome:<14}→ Acurácia: {accs[1]*100:.1f}%')
        print('-'*55)

        diff = abs(accs[0] - accs[1])
        if diff < 0.01:
            print('  🤝 Resultado: EMPATE (diferença < 1%)')
        elif accs[0] > accs[1]:
            print(f'  🏆 Vencedor: Naive Bayes (Δ = {diff*100:.1f}%)')
        else:
            print(f'  🏆 Vencedor: {nome} (Δ = {diff*100:.1f}%)')
        print('='*55)

botao.on_click(executar)

---
### Como usar
1. Execute as células **1, 2 e 3** em ordem (▶ ou `Shift+Enter`)
2. Volte à **célula 2**, selecione o algoritmo desejado no menu
3. Clique em **▶ Executar análise**
4. Os resultados e gráficos aparecem automaticamente abaixo

> Você pode trocar o algoritmo e clicar novamente para comparar resultados diferentes.

---
## Como ler a Matriz de Confusão

A **matriz de confusão** resume o desempenho de um classificador mostrando, para cada classe real, quantas vezes o modelo acertou ou errou — e em qual classe errou.

### Estrutura geral (3 classes)

|  | **Previsto: setosa** | **Previsto: versicolor** | **Previsto: virginica** |
|---|:---:|:---:|:---:|
| **Real: setosa**     | VP setosa | erro | erro |
| **Real: versicolor** | erro | VP versicolor | erro |
| **Real: virginica**  | erro | erro | VP virginica |

- **Diagonal principal** (↘) → acertos: o modelo previu a classe correta  
- **Fora da diagonal** → erros: o modelo confundiu uma classe com outra

---

### Exemplo concreto

Suponha esta matriz gerada pelo modelo:

|  | setosa | versicolor | virginica |
|---|:---:|:---:|:---:|
| **setosa**     | **10** | 0 | 0 |
| **versicolor** | 0 | **9** | 1 |
| **virginica**  | 0 | 2 | **8** |

**Como ler linha por linha (pela classe REAL):**

- Das **10 flores reais setosa** → modelo acertou **10** (100%)
- Das **10 flores reais versicolor** → acertou **9**, errou **1** (previu virginica)
- Das **10 flores reais virginica** → acertou **8**, errou **2** (previu versicolor)

> **Observação:** versicolor e virginica têm atributos mais parecidos entre si, por isso o modelo tende a confundi-las com mais frequência. setosa é geralmente classificada com 100% de acerto.

---

### Calculando a Acurácia a partir da matriz

$$\text{Acurácia} = \frac{\text{soma da diagonal}}{\text{total de amostras}} = \frac{10 + 9 + 8}{30} = \frac{27}{30} \approx 90\%$$

---

### O que observar na comparação entre os dois modelos

| Pergunta | O que olhar |
|---|---|
| Qual modelo foi mais preciso no geral? | Compare as acurácias |
| Algum modelo errou mais em uma espécie específica? | Compare linha a linha |
| Os erros ocorrem nas mesmas classes nos dois modelos? | Compare as células fora da diagonal |
| Houve empate? | Diferença de acurácia < 1% |